In [1]:
!pip install kaggle -q

from google.colab import files
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Dataset'i indir
!kaggle datasets download -d bhushandivekar/video-game-sales-and-industry-data-1980-2024
!kaggle datasets download -d thedevastator/video-game-sales-and-ratings
!kaggle datasets download -d anandshaw2001/video-game-sales

#zip dosyaları aç
import zipfile

zip_files = [
    "video-game-sales-and-industry-data-1980-2024.zip",
    "video-game-sales-and-ratings.zip",
    "video-game-sales.zip"
]

for zip_name in zip_files:
    folder_name = zip_name.replace(".zip", "")
    os.makedirs(folder_name, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(folder_name)

print("Zip dosyalari acildi.")

#hangi dosyalar geldi
print("Klasor icerigi:")
for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

#datasetleri oku
import pandas as pd

df1 = pd.read_csv("video-game-sales/vgsales.csv")
df2 = pd.read_csv("video-game-sales-and-ratings/Video_Games.csv")
df3 = pd.read_csv("video-game-sales-and-industry-data-1980-2024/Video Games Sales (1980-2024) - Raw.csv")

print("df1:", df1.shape)
print("df2:", df2.shape)
print("df3:", df3.shape)

df1.head()

#sütun adlarını oku
print(df1.columns)
print(df2.columns)
print(df3.columns)

#dataset1 userscore ve criticscore yok
df1 = df1.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year': 'year',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Global_Sales': 'global_sales',
    'JP_Sales':'jp_sales',
    'NA_Sales':'na_sales',
    'EU_Sales':'eu_sales'
})

df1['critic_score'] = pd.NA
df1['user_score'] = pd.NA

df1 = df1[['name', 'platform', 'year', 'genre', 'publisher', 'critic_score', 'user_score', 'jp_sales', 'na_sales', 'eu_sales', 'global_sales']]

#dataset2, tam veri
df2 = df2.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year_of_Release': 'year',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Global_Sales': 'global_sales',
    'JP_Sales':'jp_sales',
    'NA_Sales':'na_sales',
    'EU_Sales':'eu_sales',
    'Critic_Score': 'critic_score',
    'User_Score': 'user_score'
})

df2 = df2[['name', 'platform', 'year', 'genre', 'publisher', 'critic_score', 'user_score', 'jp_sales', 'na_sales', 'eu_sales', 'global_sales']]

#dataset3 userscore yok
df3 = df3.rename(columns={
    'title': 'name',
    'console': 'platform',
    'release_date': 'year',
    'genre': 'genre',
    'publisher': 'publisher',
    'total_sales': 'global_sales',
    'jp_sales':'jp_sales',
    'na_sales':'na_sales',
    'critic_score': 'critic_score'
})

df3['eu_sales'] = pd.NA
df3['user_score'] = pd.NA


df3 = df3[['name', 'platform', 'year', 'genre', 'publisher', 'critic_score', 'user_score', 'jp_sales', 'na_sales', 'eu_sales', 'global_sales']]






[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google.colab'

In [ ]:

final_df = pd.concat([df1, df2, df3], ignore_index=True)

In [ ]:

dup_rows = final_df[
    final_df.duplicated(
        subset=['name', 'platform', 'year', 'genre', 'publisher', 'critic_score', 'user_score', 'jp_sales', 'na_sales', 'eu_sales', 'global_sales'],
        keep=False
    )
]

dup_rows


In [ ]:
final_df['eu_sales'].isna().sum()

In [ ]:
final_df['eu_sales'].notna().sum()

In [ ]:
import pandas as pd
import numpy as np

# Year sütununu temizleme
numeric_year = pd.to_numeric(final_df['year'], errors='coerce')

date_year = pd.to_datetime(
    final_df['year'],
    errors='coerce',
    dayfirst=True
).dt.year

# Eğer değer zaten 1980-2026 arasında sayıysa onu kullan,
# değilse tarih formatından gelen yılı kullan
final_df['year'] = numeric_year.where(
    numeric_year.between(1980, 2026),
    date_year
)

# Geçersiz yılları boş yap
final_df.loc[~final_df['year'].between(1980, 2026), 'year'] = np.nan

# Year boş olanları sil
final_df = final_df.dropna(subset=['year'])

# Yılı tam sayı yap
final_df['year'] = final_df['year'].astype(int)

In [ ]:
print("Satır sayısı:", final_df.shape)
print("EU dolu satır:", final_df['eu_sales'].notna().sum())
print("EU boş satır:", final_df['eu_sales'].isna().sum())

In [ ]:
print(final_df[['year']].sample(20))

In [ ]:
import numpy as np
import pandas as pd


# 2. KARMAŞAYI ÇÖZEN KISIM:
# Sütunlarda string olarak yazılmış 'None', 'null' veya manuel eklediğin None'ları
# gerçek np.nan (Not a Number) formatına çekiyoruz.
bosluk_varyasyonlari = ['None', 'none', 'null', 'NULL', 'nan', 'NaN', ' ', 'N/A']
final_df.replace(bosluk_varyasyonlari, np.nan, inplace=True)
# Veri setinden rastgele 20 örnek çeker
final_df.sample(20)

In [ ]:
final_df = final_df.dropna(subset=['global_sales'])

In [ ]:
final_df.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

genre_counts = final_df['genre'].value_counts().reset_index()
genre_counts.columns = ['genre', 'count']

# Azalan sıraya göre sıralama
genre_counts = genre_counts.sort_values(by='count', ascending=False)

plt.figure(figsize=(15, 7))

ax = sns.barplot(
    data=genre_counts,
    x='genre',
    y='count',
    color='teal'
)

ax.bar_label(ax.containers[0], padding=3)

plt.ylim(0, genre_counts['count'].max() * 1.15)

plt.xticks(rotation=45, ha='right')
plt.title("Number of Games by Genre")
plt.xlabel("Genre")
plt.ylabel("Number of Games")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

genre_sales = final_df.groupby('genre')['global_sales'].sum().reset_index()
genre_sales = genre_sales.sort_values(by='global_sales', ascending=False)

plt.figure(figsize=(15, 7))

ax = sns.barplot(
    data=genre_sales,
    x='genre',
    y='global_sales',
    color='cornflowerblue'
)

ax.bar_label(ax.containers[0], fmt='%.2f', padding=3)

plt.ylim(0, genre_sales['global_sales'].max() * 1.15)

plt.xticks(rotation=45, ha='right')
plt.title("Total Global Sales by Genre")
plt.xlabel("Genre")
plt.ylabel("Total Global Sales")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# global_sales sayısal olsun
final_df['global_sales'] = pd.to_numeric(final_df['global_sales'], errors='coerce')

# Platformlara göre toplam satış
platform_sales = (
    final_df
    .dropna(subset=['platform', 'global_sales'])
    .groupby('platform', as_index=False)['global_sales']
    .sum()
    .sort_values(by='global_sales', ascending=False)
)

# İlk 15 platformu alalım
top_platform_sales = platform_sales.head(15)

plt.figure(figsize=(15, 7))

bars = plt.barh(
    top_platform_sales['platform'],
    top_platform_sales['global_sales'],
    color='steelblue'
)

# En yüksek değer üstte görünsün
plt.gca().invert_yaxis()

# Çubukların sonuna değer yazdırma
plt.bar_label(bars, fmt='%.2f', padding=5)

plt.title("Top 15 Platforms by Global Sales")
plt.xlabel("Total Global Sales")
plt.ylabel("Platform")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

publisher_sales = final_df.groupby('publisher')['global_sales'].sum().reset_index()
publisher_sales = publisher_sales.sort_values(by='global_sales', ascending=False).head(15)

plt.figure(figsize=(18, 7))

bars = plt.barh(
    publisher_sales['publisher'],
    publisher_sales['global_sales'],
    color='cadetblue'
)

plt.gca().invert_yaxis()

plt.bar_label(bars, fmt='%.2f', padding=5)

plt.title("Top 15 Publishers by Global Sales")
plt.xlabel("Total Global Sales")
plt.ylabel("Publisher")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

final_df['year'] = pd.to_numeric(final_df['year'], errors='coerce')

year_sales = final_df.groupby('year')['global_sales'].sum().reset_index()
year_sales = year_sales.dropna()
year_sales = year_sales.sort_values(by='year')

plt.figure(figsize=(18, 7))

plt.plot(
    year_sales['year'],
    year_sales['global_sales'],
    marker='o',
    linewidth=2,
    color='darkorange'
)

plt.title("Total Global Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Global Sales")

plt.xticks(rotation=45)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

final_df['year'] = pd.to_numeric(final_df['year'], errors='coerce')

year_sales = final_df.groupby('year')['global_sales'].sum().reset_index()
year_sales = year_sales.dropna()
year_sales = year_sales.sort_values(by='year')

plt.figure(figsize=(15, 7))

plt.fill_between(
    year_sales['year'],
    year_sales['global_sales'],
    alpha=0.4,
    color='mediumseagreen'
)

plt.plot(
    year_sales['year'],
    year_sales['global_sales'],
    marker='o',
    linewidth=2,
    color='seagreen'
)

plt.title("Total Global Sales Trend by Year")
plt.xlabel("Year")
plt.ylabel("Total Global Sales")

plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Genre bazlı toplam satışlar
genre_sales = final_df.groupby('genre')[['na_sales', 'eu_sales', 'jp_sales']].sum()

# Grafik
genre_sales.plot(kind='bar', figsize=(14,7))
plt.title("Bölgelere Göre Tür Bazlı Toplam Satışlar")
plt.xlabel("Tür (Genre)")
plt.ylabel("Satış (Milyon)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Yıl bazlı toplam satış
yearly_sales = final_df.groupby('year')[['na_sales', 'eu_sales', 'jp_sales']].sum()

# Grafik
yearly_sales.plot(figsize=(14,7))
plt.title("Yıllara Göre Bölgesel Satış Trendi")
plt.xlabel("Yıl")
plt.ylabel("Toplam Satış (Milyon)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
regions = ['na_sales', 'eu_sales', 'jp_sales']

plt.figure(figsize=(14,10))

for i, region in enumerate(regions, 1):
    plt.subplot(3,1,i)
    final_df.groupby('year')[region].sum().plot()
    plt.title(f"{region.upper()} Yıllara Göre Satış")
    plt.xlabel("Yıl")
    plt.ylabel("Satış")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Bölgesel satış sütunları
region_cols = ['na_sales', 'eu_sales', 'jp_sales']

# Sayısala çevir
for col in region_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

# Gerekli sütunlarda boş olanları temizle
temp_df = final_df[['platform', 'genre'] + region_cols].copy()
temp_df[region_cols] = temp_df[region_cols].fillna(0)

# Uzun formata çevir
long_df = temp_df.melt(
    id_vars=['platform', 'genre'],
    value_vars=region_cols,
    var_name='region',
    value_name='sales'
)

# Bölge isimlerini daha güzel yap
long_df['region'] = long_df['region'].replace({
    'na_sales': 'North America',
    'eu_sales': 'Europe',
    'jp_sales': 'Japan'
})

In [ ]:
platform_pref = (
    long_df
    .groupby(['region', 'platform'])['sales']
    .sum()
    .reset_index()
)

regions = ['North America', 'Europe', 'Japan']

for region in regions:
    top_platforms = (
        platform_pref[platform_pref['region'] == region]
        .sort_values(by='sales', ascending=False)
        .head(20)
    )

    plt.figure(figsize=(12, 6))
    plt.bar(top_platforms['platform'], top_platforms['sales'])
    plt.title(f'Top 20 Platform Preferences in {region}')
    plt.xlabel('Platform')
    plt.ylabel('Total Sales')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
genre_pref = (
    long_df
    .groupby(['region', 'genre'])['sales']
    .sum()
    .reset_index()
)

for region in regions:
    top_genres = (
        genre_pref[genre_pref['region'] == region]
        .sort_values(by='sales', ascending=False)
    )

    plt.figure(figsize=(12, 6))
    plt.bar(top_genres['genre'], top_genres['sales'])
    plt.title(f'Genre Preferences in {region}')
    plt.xlabel('Genre')
    plt.ylabel('Total Sales')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

cute_colors = ['#FF8FAB', '#7BDFF2', '#B2F7A5']
# pembe, açık mavi, yeşil

region_order = ['North America', 'Europe', 'Japan']

In [ ]:
genre_pivot = (
    long_df
    .groupby(['genre', 'region'])['sales']
    .sum()
    .unstack()
    .fillna(0)
)

# Bölge sütunlarını düzgün sırala
genre_pivot = genre_pivot.reindex(columns=region_order)

# Toplam satışa göre azalan sırala
genre_pivot['total_sales'] = genre_pivot.sum(axis=1)
genre_pivot = genre_pivot.sort_values(by='total_sales', ascending=False)
genre_pivot = genre_pivot.drop(columns='total_sales')

# Grafik
genre_pivot.plot(
    kind='bar',
    figsize=(18, 7),
    color=cute_colors,
    edgecolor='black'
)

plt.title('Regional Genre Preferences', fontsize=14)
plt.xlabel('Genre')
plt.ylabel('Total Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
platform_total = (
    long_df
    .groupby('platform')['sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

platform_pivot = (
    long_df[long_df['platform'].isin(platform_total)]
    .groupby(['platform', 'region'])['sales']
    .sum()
    .unstack()
    .fillna(0)
)

# Bölge sütunlarını düzgün sırala
platform_pivot = platform_pivot.reindex(columns=region_order)

# Toplam satışa göre azalan sırala
platform_pivot['total_sales'] = platform_pivot.sum(axis=1)
platform_pivot = platform_pivot.sort_values(by='total_sales', ascending=False)
platform_pivot = platform_pivot.drop(columns='total_sales')

# Grafik
platform_pivot.plot(
    kind='bar',
    figsize=(14, 7),
    color=cute_colors,
    edgecolor='black'
)

plt.title('Regional Platform Preferences (Top 10 Platforms)', fontsize=14)
plt.xlabel('Platform')
plt.ylabel('Total Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
model_df = final_df.copy()

In [ ]:
import pandas as pd
import numpy as np

sales_cols = ['global_sales', 'na_sales', 'eu_sales', 'jp_sales']

for col in sales_cols:
   model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

In [ ]:
model_df['global_sales'].describe()

In [ ]:
# model_df['global_sales'].quantile([0.33, 0.66])

In [ ]:
# q1 = model_df['global_sales'].quantile(0.33)
# q2 = model_df['global_sales'].quantile(0.66)

# def sales_class(x):
    #  if x <= q1:
        #  return 'low'
    #  elif x <= q2:
        #  return 'medium'
    #  else:
        #  return 'high'

# model_df['sales_class'] = model_df['global_sales'].apply(sales_class)

In [ ]:
model_df['sales_class'] = pd.qcut(
    model_df['global_sales'],
    q=3,
    labels=['low', 'medium', 'high']
)

In [ ]:
model_df[['name', 'global_sales', 'sales_class']].head(10)

In [ ]:
model_df['sales_class'].value_counts()

In [ ]:
model_df['sales_class'].value_counts(normalize=True) * 100

In [ ]:
import pandas as pd
import numpy as np

model_df['year'] = pd.to_numeric(model_df['year'], errors='coerce')
model_df['critic_score'] = pd.to_numeric(model_df['critic_score'], errors='coerce')
model_df['user_score'] = pd.to_numeric(model_df['user_score'], errors='coerce')
model_df['global_sales'] = pd.to_numeric(model_df['global_sales'], errors='coerce')

In [ ]:
top_publishers = model_df['publisher'].value_counts().head(15).index

model_df['publisher_grouped'] = model_df['publisher'].apply(
    lambda x: x if x in top_publishers else 'Other'
)

*Modelde kullanacağımız özellikler*

In [ ]:
features = [
    'platform',
    'year',
    'genre',
    'publisher_grouped'
]

X = model_df[features]
y = model_df['sales_class']

In [ ]:
model_df['critic_score'].isna().mean() * 100
# user ve critic score çıkartıldı ilk modelden

In [ ]:
X = X.copy()

X['platform'] = X['platform'].fillna('Unknown')
X['genre'] = X['genre'].fillna('Unknown')
X['publisher_grouped'] = X['publisher_grouped'].fillna('Unknown')

X['year'] = X['year'].fillna(X['year'].median())

*one-hot encoding*

In [ ]:
X_encoded = pd.get_dummies(
    X,
    columns=['platform', 'genre', 'publisher_grouped'],
    drop_first=True
)

In [ ]:
X_encoded.head()

In [ ]:
X_encoded.shape

In [ ]:
X_encoded.isna().sum().sum()

*train-test split + baseline model aşaması*

In [ ]:
y = model_df['sales_class']

In [ ]:
y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
test_size=0.2

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

dummy_model = DummyClassifier(strategy='most_frequent')

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

print("Dummy Model Accuracy:", accuracy_score(y_test, y_pred_dummy))
print(classification_report(y_test, y_pred_dummy))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=12
)

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100, # Increased n_estimators for better performance
    random_state=42, # Added random_state for reproducibility
    max_depth=12,
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Denenecek parametreler
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 10, 12, 15, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Ana model
rf_model = RandomForestClassifier(
    random_state=42
)

# GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

# Eğitim
grid_search.fit(X_train, y_train)

# En iyi modeli al
best_rf_model = grid_search.best_estimator_

# Test verisiyle tahmin yap
y_pred_rf = best_rf_model.predict(X_test)

# Sonuçları yazdır
print("En iyi parametreler:")
print(grid_search.best_params_)

print("En iyi Cross Validation skoru:")
print(grid_search.best_score_)

print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
print("Dummy Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
)

# Eğitim
random_search.fit(X_train, y_train)

# En iyi model
best_rf_model = random_search.best_estimator_

# Test tahmini
y_pred_rf = best_rf_model.predict(X_test)

# Çıktılar
print("En iyi parametreler:")
print(random_search.best_params_)

print("\nEn iyi Cross Validation skoru:")
print(random_search.best_score_)

print("\nRandom Forest Test Accuracy:")
print(accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Parametre aralığı
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Model
rf_model = RandomForestClassifier(random_state=42)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=15,              # biraz artırdık (daha iyi sonuç için)
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
)

# Eğitim
random_search.fit(X_train, y_train)

# En iyi model
best_rf_model = random_search.best_estimator_

# Test tahmini
y_pred_rf = best_rf_model.predict(X_test)

# Çıktılar (SENİN FORMATINDA)
print("En iyi parametreler:")
print(random_search.best_params_)

print("\nEn iyi Cross Validation skoru:")
print(random_search.best_score_)

print("\nRandom Forest Test Accuracy:")
print(accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Model
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
    )

# Eğitim
gb_model.fit(X_train, y_train)

# Tahmin
y_pred_gb = gb_model.predict(X_test)

# Çıktılar (RF ile aynı format)
print("Gradient Boosting Test Accuracy:")
print(accuracy_score(y_test, y_pred_gb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
lr_model = LogisticRegression(max_iter=1000)

# Eğitim
lr_model.fit(X_train_scaled, y_train)

# Tahmin
y_pred_lr = lr_model.predict(X_test_scaled)

# Çıktı (RF formatında)
print("Logistic Regression Test Accuracy:")
print(accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
threshold_df = model_df.copy()

In [ ]:
threshold_df['global_sales'] = pd.to_numeric(threshold_df['global_sales'], errors='coerce')
threshold_df = threshold_df.dropna(subset=['global_sales'])

sales_threshold = threshold_df['global_sales'].quantile(0.75)

threshold_df['investment_class'] = threshold_df['global_sales'].apply(
    lambda x: 'good_investment' if x >= sales_threshold else 'risky_investment'
)

threshold_df['investment_class'].value_counts()

In [ ]:
features = [
    'platform',
    'year',
    'genre',
    'publisher_grouped'
]

X_binary = threshold_df[features]
y_binary = threshold_df['investment_class']

In [ ]:
X_binary = X_binary.copy()

X_binary['platform'] = X_binary['platform'].fillna('Unknown')
X_binary['genre'] = X_binary['genre'].fillna('Unknown')
X_binary['publisher_grouped'] = X_binary['publisher_grouped'].fillna('Unknown')
X_binary['year'] = X_binary['year'].fillna(X_binary['year'].median())

In [ ]:
X_binary_encoded = pd.get_dummies(
    X_binary,
    columns=['platform', 'genre', 'publisher_grouped'],
    drop_first=True
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_binary_encoded,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_binary = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=12,
    class_weight = 'balanced'
)

rf_binary.fit(X_train_bin, y_train_bin)

y_pred_bin = rf_binary.predict(X_test_bin)

print("Binary Model Accuracy:", accuracy_score(y_test_bin, y_pred_bin))
print(classification_report(y_test_bin, y_pred_bin))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

dummy_bin = DummyClassifier(strategy='most_frequent')

dummy_bin.fit(X_train_bin, y_train_bin)
y_pred_dummy_bin = dummy_bin.predict(X_test_bin)

print("Dummy Accuracy:", accuracy_score(y_test_bin, y_pred_dummy_bin))
print(classification_report(y_test_bin, y_pred_dummy_bin))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score, f1_score

rf_binary_balanced = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=12,
    class_weight='balanced'
)

rf_binary_balanced.fit(X_train_bin, y_train_bin)

y_pred_balanced = rf_binary_balanced.predict(X_test_bin)

print("Accuracy:", accuracy_score(y_test_bin, y_pred_balanced))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_bin, y_pred_balanced))
print("F1 Macro:", f1_score(y_test_bin, y_pred_balanced, average='macro'))
print(classification_report(y_test_bin, y_pred_balanced, zero_division=0))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, precision_score

good_index = list(rf_binary_balanced.classes_).index('good_investment')
good_probs = rf_binary_balanced.predict_proba(X_test_bin)[:, good_index]

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred_threshold = np.where(
        good_probs >= threshold,
        'good_investment',
        'risky_investment'
    )

    print("Threshold:", threshold)
    print("Accuracy:", accuracy_score(y_test_bin, y_pred_threshold))
    print("Balanced Accuracy:", balanced_accuracy_score(y_test_bin, y_pred_threshold))
    print("F1 Macro:", f1_score(y_test_bin, y_pred_threshold, average='macro'))
    print("Good Precision:", precision_score(y_test_bin, y_pred_threshold, pos_label='good_investment'))
    print("Good Recall:", recall_score(y_test_bin, y_pred_threshold, pos_label='good_investment'))
    print("-" * 40)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix

good_index = list(rf_binary_balanced.classes_).index('good_investment')
good_probs = rf_binary_balanced.predict_proba(X_test_bin)[:, good_index]

selected_threshold = 0.45

y_pred_final = np.where(
    good_probs >= selected_threshold,
    'good_investment',
    'risky_investment'
)

print("Selected Threshold:", selected_threshold)
print("Accuracy:", accuracy_score(y_test_bin, y_pred_final))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_bin, y_pred_final))
print("F1 Macro:", f1_score(y_test_bin, y_pred_final, average='macro'))
print("Good Precision:", precision_score(y_test_bin, y_pred_final, pos_label='good_investment'))
print("Good Recall:", recall_score(y_test_bin, y_pred_final, pos_label='good_investment'))
print(classification_report(y_test_bin, y_pred_final))

In [ ]:
cm = confusion_matrix(y_test_bin, y_pred_final)

cm_df = pd.DataFrame(
    cm,
    index=['Actual good_investment', 'Actual risky_investment'],
    columns=['Predicted good_investment', 'Predicted risky_investment']
)

cm_df

In [ ]:
selected_threshold = 0.60

y_pred_final = np.where(
    good_probs >= selected_threshold,
    'good_investment',
    'risky_investment'
)

print("Selected Threshold:", selected_threshold)
print("Accuracy:", accuracy_score(y_test_bin, y_pred_final))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_bin, y_pred_final))
print("F1 Macro:", f1_score(y_test_bin, y_pred_final, average='macro'))
print("Good Precision:", precision_score(y_test_bin, y_pred_final, pos_label='good_investment'))
print("Good Recall:", recall_score(y_test_bin, y_pred_final, pos_label='good_investment'))
print(classification_report(y_test_bin, y_pred_final))

In [ ]:
def recommendation_level(prob):
    if prob >= 0.60:
        return 'recommend_investment'
    elif prob >= 0.45:
        return 'needs_review'
    else:
        return 'not_recommended'


In [ ]:
recommendation_results = pd.DataFrame({
    'actual_class': y_test_bin.values,
    'good_investment_probability': good_probs
})

recommendation_results['recommendation_level'] = recommendation_results['good_investment_probability'].apply(recommendation_level)

recommendation_results.head(10)


In [ ]:
recommendation_results['recommendation_level'].value_counts()

In [ ]:
!pip install catboost -q

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

features = [
    'platform',
    'year',
    'genre',
    'publisher_grouped'
]

target = 'investment_class'

cat_features = [
    'platform',
    'genre',
    'publisher_grouped'
]

# Sadece gerekli sütunları al
catboost_df = threshold_df[features + [target]].copy()

# Target boşsa sil
catboost_df = catboost_df.dropna(subset=[target])

# Kategorik sütunlardaki NaN değerleri doldur ve string'e çevir
for col in cat_features:
    catboost_df[col] = catboost_df[col].fillna("Unknown").astype(str)

# year sayısal kalmalı
catboost_df['year'] = pd.to_numeric(catboost_df['year'], errors='coerce')
catboost_df['year'] = catboost_df['year'].fillna(catboost_df['year'].median())

X_binary = catboost_df[features]
y_binary = catboost_df[target]

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_binary,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='F1',
    random_seed=42,
    auto_class_weights='Balanced',
    verbose=100
)

cat_model.fit(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features,
    eval_set=(X_test_cat, y_test_cat),
    use_best_model=True
)

y_pred_cat = cat_model.predict(X_test_cat)

print("Accuracy:", accuracy_score(y_test_cat, y_pred_cat))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_cat, y_pred_cat))
print("F1 Macro:", f1_score(y_test_cat, y_pred_cat, average='macro'))
print("\nClassification Report:\n", classification_report(y_test_cat, y_pred_cat))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_cat, y_pred_cat))

In [ ]:
y_proba = cat_model.predict_proba(X_test_cat)

cat_model.classes_

In [ ]:
good_proba = y_proba[:, 0]

In [ ]:
threshold = 0.60


y_pred_threshold = [
    'good_investment' if prob >= threshold else 'risky_investment'
    for prob in good_proba
]

print("Accuracy:", accuracy_score(y_test_cat, y_pred_threshold))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_cat, y_pred_threshold))
print("F1 Macro:", f1_score(y_test_cat, y_pred_threshold, average='macro'))
print(classification_report(y_test_cat, y_pred_threshold))
print(confusion_matrix(y_test_cat, y_pred_threshold))

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score

y_proba = cat_model.predict_proba(X_test_cat)

print(cat_model.classes_)

In [ ]:
good_proba = y_proba[:, 0]

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

for threshold in thresholds:
    y_pred_threshold = [
        'good_investment' if prob >= threshold else 'risky_investment'
        for prob in good_proba
    ]

    print("\nThreshold:", threshold)
    print("Accuracy:", accuracy_score(y_test_cat, y_pred_threshold))
    print("Balanced Accuracy:", balanced_accuracy_score(y_test_cat, y_pred_threshold))
    print("F1 Macro:", f1_score(y_test_cat, y_pred_threshold, average='macro'))
    print("Good Precision:", precision_score(y_test_cat, y_pred_threshold, pos_label='good_investment'))
    print("Good Recall:", recall_score(y_test_cat, y_pred_threshold, pos_label='good_investment'))
    print(confusion_matrix(y_test_cat, y_pred_threshold))

*Final Model: CatBoostClassifier*
Decision Threshold: 0.60
Target: investment_class
Classes: good_investment / risky_investment
Ana metrik: F1 Macro
Yardımcı metrikler: Balanced Accuracy, good_investment precision, good_investment recall

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_cv = catboost_df[features].copy()
y_cv = catboost_df[target].values

# Kategorikleri string'e çevir (category dtype değil)
for col in cat_features:
    X_cv[col] = X_cv[col].astype(str)

X_cv = X_cv.reset_index(drop=True)

results = {'accuracy': [], 'balanced_accuracy': [], 'f1_macro': []}

for fold, (train_idx, val_idx) in enumerate(cv.split(X_cv, y_cv)):
    X_tr, X_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
    y_tr, y_val = y_cv[train_idx], y_cv[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='Logloss',
        eval_metric='F1',
        random_seed=42,
        auto_class_weights='Balanced',
        verbose=0
    )

    model.fit(X_tr, y_tr, cat_features=cat_features)
    y_pred = model.predict(X_val)

    results['accuracy'].append(accuracy_score(y_val, y_pred))
    results['balanced_accuracy'].append(balanced_accuracy_score(y_val, y_pred))
    results['f1_macro'].append(f1_score(y_val, y_pred, average='macro'))

    print(f"Fold {fold+1} → Acc: {results['accuracy'][-1]:.4f} | Bal.Acc: {results['balanced_accuracy'][-1]:.4f} | F1: {results['f1_macro'][-1]:.4f}")

print("\n=== 5-Fold Cross Validation Sonuçları ===")
for metric, vals in results.items():
    print(f"{metric:20s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np

In [ ]:
rec_features = ['platform', 'genre', 'publisher_grouped', 'year']

recommend_df = catboost_df.copy()

recommend_df = recommend_df.dropna(subset=rec_features)

X_rec = recommend_df[rec_features]

rec_features = [
    'Platform',
    'Genre',
    'Publisher',
    'Year',
    'Critic_Score',
    'User_Score',
    'Global_Sales_Class'
]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
categorical_cols = ['platform', 'genre', 'publisher_grouped']
numeric_cols = ['year']

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

In [ ]:
categorical_cols = ['platform', 'genre', 'publisher_grouped']
numeric_cols = ['year']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

X_processed = preprocessor.fit_transform(X_rec)

In [ ]:
print(type(X_processed))
print(X_processed.shape)

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
nn_model = NearestNeighbors(
    n_neighbors=6,
    metric='cosine'
)

In [ ]:
nn_model.fit(X_processed)

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    # Use indices to fetch full data from threshold_df
    original_indices = recommend_df.index[indices[0]]
    recommendations = threshold_df.loc[original_indices].copy()

    recommendations['Similarity'] = 1 - distances[0]

    return recommendations[[
        'name',
        'platform',
        'genre',
        'publisher_grouped',
        'year',
        'global_sales',
        'Similarity'
    ]]

In [ ]:
recommend_games(
    platform='PS4',
    genre='Action',
    publisher='Ubisoft',
    year=2016
)

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    # Use indices to fetch full data from threshold_df
    original_indices = recommend_df.index[indices[0]]
    recommendations = threshold_df.loc[original_indices].copy()

    recommendations['Similarity'] = 1 - distances[0]

    # Risk seviyesi
    recommendations['Risk_Level'] = np.where(
        recommendations['global_sales'] > 10,
        'Low Risk',
        np.where(
            recommendations['global_sales'] > 3,
            'Medium Risk',
            'High Risk'
        )
    )

    # Generate reasons
    def get_reasons(row):
        reasons = []
        if platform == row['platform']:
            reasons.append("Same platform")
        if genre == row['genre']:
            reasons.append("Same genre")
        if publisher == row['publisher_grouped']:
            reasons.append("Same publisher")
        if abs(year - row['year']) <= 2:
            reasons.append("Similar release period")
        print("Why Recommended:")
        for reason in reasons:
          print("-", reason)
        return ", ".join(reasons)

    recommendations['Reasons'] = recommendations.apply(get_reasons, axis=1)

    return recommendations[[
        'name',
        'platform',
        'genre',
        'publisher_grouped',
        'year',
        'global_sales',
        'Similarity',
        'Risk_Level',
        'Reasons'
    ]]

In [ ]:
recommend_games(
    platform='PS4',
    genre='Action',
    publisher='Ubisoft',
    year=2016
)

In [ ]:
print((threshold_df['global_sales'] > 10).mean())

In [ ]:
# The logic has been moved into the recommend_games function.
# Let's test it out:
recommend_games(
    platform='PS4',
    genre='Action',
    publisher='Ubisoft',
    year=2016
)

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    recommendations = recommend_df.iloc[indices[0]].copy()

    recommendations['Similarity'] = 1 - distances[0]

    # ✅ Investment Score burada olmalı
    recommendations['Investment_Score'] = (
        recommendations['Similarity'] * 50 +
        np.clip(recommendations['global_sales'], 0, 10) * 5
    )

    return recommendations

In [ ]:
print(recommend_df.columns)

In [ ]:
recommend_df = recommend_df.merge(
    model_df[['platform', 'year', 'genre', 'publisher_grouped', 'global_sales']],
    on=['platform', 'year', 'genre', 'publisher_grouped'],
    how='left'
)

In [ ]:
print(recommend_df['global_sales'].isna().mean())

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)
    distances, indices = nn_model.kneighbors(input_processed)

    recommendations = recommend_df.iloc[indices[0]].copy()
    recommendations['Similarity'] = 1 - distances[0]

    recommendations['Investment_Score'] = (
        recommendations['Similarity'] * 50 +
        np.clip(recommendations['global_sales'], 0, 10) * 5
    )

    return recommendations.sort_values(by='Investment_Score', ascending=False)

In [ ]:
recs = recommend_games('PS4', 'Action', 'Ubisoft', 2016)

recs[['platform','genre','publisher_grouped','year','Similarity','global_sales','Investment_Score']]

In [ ]:
recommendations = recommend_df.iloc[indices[0]].copy()

recommendations = recommendations.iloc[1:]  # 🔥 ilk satırı at

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    recommendations = recommend_df.iloc[indices[0]].copy()

    # 🔥 kendisini listeden çıkar
    recommendations = recommendations.iloc[1:]

    # similarity fix
    recommendations['Similarity'] = 1 / (1 + distances[0][1:])

    recommendations['Investment_Score'] = (
        recommendations['Similarity'] * 50 +
        np.clip(recommendations['global_sales'], 0, 10) * 5
    )

    return recommendations.sort_values(by='Investment_Score', ascending=False)

In [ ]:
recs = recommend_games('PS4', 'Action', 'Ubisoft', 2016)

recs[['platform','genre','publisher_grouped','year','Similarity','global_sales','Investment_Score']]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors

rec_features = ['platform', 'genre', 'publisher_grouped', 'year']

X_rec = recommend_df[rec_features].copy()

categorical_cols = ['platform', 'genre', 'publisher_grouped']
numeric_cols = ['year']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', MinMaxScaler(), numeric_cols)
])

X_processed = preprocessor.fit_transform(X_rec)

nn_model = NearestNeighbors(
    n_neighbors=6,
    metric='euclidean'
)

nn_model.fit(X_processed)

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    recommendations = recommend_df.iloc[indices[0]].copy()

    recommendations['Distance'] = distances[0]
    recommendations['Similarity'] = 1 / (1 + recommendations['Distance'])

    recommendations['Investment_Score'] = (
        recommendations['Similarity'] * 50 +
        np.clip(recommendations['global_sales'], 0, 10) * 5
    )

    return recommendations.sort_values(by='Investment_Score', ascending=False)

In [ ]:
recs = recommend_games('PS4', 'Action', 'Ubisoft', 2016)

recs[['platform','genre','publisher_grouped','year','Distance','Similarity','global_sales','Investment_Score']]

In [ ]:
recommend_df = recommend_df.merge(
    model_df[['platform', 'year', 'genre', 'publisher_grouped', 'name']],
    on=['platform', 'year', 'genre', 'publisher_grouped'],
    how='left'
)

In [ ]:
rec_features = ['platform', 'genre', 'publisher_grouped', 'year']

X_rec = recommend_df[rec_features]
X_processed = preprocessor.fit_transform(X_rec)
nn_model.fit(X_processed)

In [ ]:
recommend_df

In [ ]:
recs = recommend_games('PC','Platform','THQ',2004)

recs[['platform','genre','publisher_grouped','year','Similarity','global_sales','Investment_Score']]

In [ ]:
X_rec = recommend_df[rec_features]
X_processed = preprocessor.fit_transform(X_rec)
nn_model.fit(X_processed)

In [ ]:
def recommend_games(platform, genre, publisher, year):

    input_df = pd.DataFrame([{
        'platform': platform,
        'genre': genre,
        'publisher_grouped': publisher,
        'year': year
    }])

    input_processed = preprocessor.transform(input_df)

    distances, indices = nn_model.kneighbors(input_processed)

    recommendations = recommend_df.iloc[indices[0]].copy()

    # kendisini çıkar
    recommendations = recommendations.iloc[1:]
    distances = distances[0][1:]

    # similarity
    recommendations['Similarity'] = 1 / (1 + distances)

    # investment score
    recommendations['Investment_Score'] = (
        recommendations['Similarity'] * 50 +
        np.clip(recommendations['global_sales'], 0, 10) * 5
    )

    # 🔥 WHY RECOMMENDED
    def generate_reason(row):
        reasons = []

        if row['genre'] == genre:
            reasons.append("Same genre")

        if row['platform'] == platform:
            reasons.append("Same platform")

        if row['publisher_grouped'] == publisher:
            reasons.append("Same publisher")

        if abs(row['year'] - year) <= 2:
            reasons.append("Close release year")

        return ", ".join(reasons)

    recommendations['Why_Recommended'] = recommendations.apply(generate_reason, axis=1)

    return recommendations.sort_values(by='Investment_Score', ascending=False)

In [ ]:
recs = recommend_games('PC', 'Platform', 'THQ', 2004)

recs[['platform','genre','publisher_grouped','year','Similarity','global_sales','Investment_Score','Why_Recommended']]